# Train the Aircraft Vision model on Kaggle (free, runs in the background)

Settings (right-hand panel): **Accelerator → GPU T4 x2**, **Internet → On**.
Then **Save Version → Save & Run All (Commit) → Save**. You can close the browser; it runs on its own (about 3–5 hours).
When it's done, open the notebook's **Output** tab and download `aircraft_model.zip`.

In [ ]:
import os, subprocess, shutil, json

def run(cmd):
    """Run a command, show its output live, and stop with a clear message if it fails."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'This step failed (exit code {p.returncode}). Copy the last lines above and send them to Claude.')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. In the right-hand panel set Accelerator → GPU T4 x2, then run again.')
print('GPU:', torch.cuda.get_device_name(0))

run('git clone -q https://github.com/Themagicianleo/aircraft-vision.git /tmp/aircraft-vision || git -C /tmp/aircraft-vision pull -q')
os.chdir('/tmp/aircraft-vision/training')
run('pip install -q timm onnx onnxruntime onnxscript')

DATA = '/tmp/av_data'                # big scratch data, not kept
OUT = '/kaggle/working'              # kept: shows up in the Output tab

## Photos 1/2: FGVC-Aircraft (100 types)

In [ ]:
run(f'python -c "from av.data import prepare_fgvc; prepare_fgvc(\'{DATA}\', workers=4)"')

## Photos 2/2: Wikimedia Commons (modern types + more airliner photos)

In [ ]:
try:
    run(f'python fetch_commons.py --out {DATA}/commons --per-class 250')
except RuntimeError as e:
    print('WARNING: Commons download failed, training without it.', e)
n = sum(len(f) for _, _, f in os.walk(f'{DATA}/commons'))
print(f'Commons photos ready: {n}')

## Train

In [ ]:
extra = f'--extra-data {DATA}/commons' if os.path.exists(f'{DATA}/commons') else ''
run(f'python train.py --data-root {DATA} {extra} --out {OUT}/run --resume --workers 4 --epochs 40 --img-size 320')

## Export and measure accuracy

In [ ]:
run(f'python export.py --checkpoint {OUT}/run/last.pt --out {OUT}/exported --int8 --calib-dir {DATA}/fgvc_prepared/images')
run(f'python evaluate.py --onnx {OUT}/exported/aircraft.onnx --data-root {DATA} {extra}')
run(f'python evaluate.py --onnx {OUT}/exported/aircraft_int8.onnx --labels {OUT}/exported/labels.json --data-root {DATA} {extra}')

## Package `aircraft_model.zip` (download it from the Output tab)

In [ ]:
pkg = '/tmp/aircraft_model'
os.makedirs(pkg, exist_ok=True)
shutil.copy(f'{OUT}/exported/aircraft.onnx', f'{pkg}/aircraft.onnx')
shutil.copy(f'{OUT}/exported/labels.json', f'{pkg}/labels.json')
if os.path.exists(f'{DATA}/commons/CREDITS.csv'):
    shutil.copy(f'{DATA}/commons/CREDITS.csv', f'{pkg}/PHOTO_CREDITS.csv')  # CC licences require crediting authors
shutil.make_archive(f'{OUT}/aircraft_model', 'zip', pkg)
os.remove(f'{OUT}/run/last.pt.tmp') if os.path.exists(f'{OUT}/run/last.pt.tmp') else None
print('Done! Download aircraft_model.zip from the Output tab.')